#### Extraction des caractéristiques avec model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt

Ce script réalise l’extraction de caractéristiques compactes (4096 dimensions) à partir d’images satellites à l’aide d’un modèle Fully Convolutional basé sur VGG16. Voici les étapes principales expliquées :

Chargement des données et préparation des chemins : Le script commence par définir les chemins des fichiers nécessaires, notamment le modèle sauvegardé (model_path), le répertoire contenant les images de test (test_image_dir), le fichier CSV contenant les métadonnées associées aux images (csv_path), et l'emplacement où sauvegarder le fichier final avec les caractéristiques extraites (output_path).

Définition du modèle Fully Convolutional : Une classe VGGFullyConv est définie pour adapter l'architecture VGG16. Les couches convolutives (self.features) extraient des caractéristiques riches, et les couches de classification convolutive (self.classifier) compressent ces caractéristiques à 4096 dimensions via deux convolutions 1x1. Une réduction spatiale moyenne (mean(dim=(2, 3))) est ensuite appliquée pour condenser les informations spatiales.

Chargement du modèle et mise en mode évaluation : Le modèle est instancié et ses poids sont chargés depuis un fichier pré-entraîné. Il est ensuite transféré sur un GPU ou CPU selon la disponibilité, et configuré pour le mode évaluation (désactivation du calcul des gradients).

Transformation des images : Un pipeline de transformation est défini pour normaliser les images en entrée afin qu'elles soient compatibles avec VGG16. Les images sont redimensionnées à 224x224 pixels, converties en tenseurs, et normalisées en utilisant les moyennes et écarts-types des canaux RGB du jeu ImageNet.

Vérification des dimensions des caractéristiques : Une image factice (aléatoire) est passée dans le modèle pour vérifier que la sortie après réduction est de taille [1, 4096]. Cette étape permet de s'assurer que le modèle fonctionne comme prévu.

Extraction des caractéristiques compactes : Pour chaque image du fichier CSV, le script :

Vérifie si le fichier image existe dans le répertoire.
Charge et transforme l’image avec le pipeline défini.
Passe l’image transformée dans le modèle Fully Convolutional pour extraire un vecteur de caractéristiques compactes de taille 4096.
Si une image est manquante ou corrompue, un vecteur de zéros est utilisé par défaut.
Fusion des caractéristiques avec le fichier CSV : Les vecteurs de caractéristiques extraits pour chaque image sont combinés avec les métadonnées existantes du fichier CSV. Chaque vecteur (4096 dimensions) est ajouté comme nouvelles colonnes, nommées feature_0 à feature_4095.

Sauvegarde des résultats : Le DataFrame final, contenant les métadonnées et les caractéristiques extraites, est sauvegardé dans un fichier CSV à l’emplacement spécifié (output_path). Le script confirme la sauvegarde par un message.

En résumé, ce script transforme des images satellites en vecteurs de caractéristiques compactes de 4096 dimensions, facilitant ainsi leur utilisation pour des tâches d’analyse ou de modélisation ultérieures. Les étapes de gestion des erreurs et de fusion garantissent la robustesse et l’intégrité des résultats.

In [1]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models

# Chemins des fichiers et répertoires
model_path = r"E:\wealth_predict\models\model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt"
test_image_dir = r"E:\wealth_predict\data\downloaded\Image_satellite_EHCVM_2018_Zoom_18_Image_2024"
csv_path = r"E:\wealth_predict\data\processed_csv\DataCIV3_images_with_new_columns.csv"
output_path = r"E:\wealth_predict\data\processed_csv\DataCIV3_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv"

# Définir l'architecture du modèle Fully Convolutional
class VGGFullyConv(nn.Module):
    def __init__(self, num_classes=4):
        super(VGGFullyConv, self).__init__()
        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, num_classes, kernel_size=1)  # Nombre de classes
        )

    def forward(self, x):
        x = self.features(x)  # Extraction des caractéristiques convolutives
        x = self.classifier[:2](x)  # Appliquer les convolutions 1x1 jusqu'à la production de 4096 caractéristiques
        return x.mean(dim=(2, 3))  # Réduction spatiale (moyenne sur hauteur et largeur)

# Charger le DataFrame contenant les métadonnées
df = pd.read_csv(csv_path)

# Charger le modèle sauvegardé
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VGGFullyConv(num_classes=4).to(device)  # Charger l'architecture
model.load_state_dict(torch.load(model_path), strict=False)  # Charger les poids
model.eval()  # Mettre le modèle en mode évaluation

# Transformation des images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Fonction pour charger une image
def load_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        return transform(image)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

# Vérification de la taille des caractéristiques
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_features = model(dummy_image)  # Passer une image factice dans le modèle
print(f"Dimensions des caractéristiques compactes : {dummy_features.shape}")  # Résultat attendu : [1, 4096]

# Extraction des caractéristiques compactes
def extract_image_features(model, image_tensor):
    image_tensor = image_tensor.unsqueeze(0).to(device)  # Ajouter une dimension pour le batch
    with torch.no_grad():
        features = model(image_tensor)  # Passer par le modèle Fully Convolutional
    return features.cpu().numpy().flatten()  # Retourner les caractéristiques compactes comme tableau numpy

# Liste pour stocker les caractéristiques des images
features_list = []
for idx, row in df.iterrows():
    image_name = row['nom de l\'image']
    image_path = os.path.join(test_image_dir, image_name)

    if os.path.exists(image_path):  # Vérifier si l'image existe
        image_tensor = load_image(image_path)
        if image_tensor is not None:
            features = extract_image_features(model, image_tensor)
        else:
            features = np.zeros(4096)  # Vecteur par défaut si erreur
    else:
        print(f"Image introuvable : {image_name}")
        features = np.zeros(4096)  # Vecteur par défaut si l'image n'existe pas

    features_list.append(features)

# Conversion des caractéristiques en DataFrame
features_df = pd.DataFrame(features_list, columns=[f"feature_{i}" for i in range(4096)])

# Combiner les caractéristiques avec le DataFrame original
df_with_features = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# Sauvegarder le DataFrame final
df_with_features.to_csv(output_path, index=False)
print(f"DataFrame avec caractéristiques compactes (4096) sauvegardé sous : {output_path}")


Dimensions des caractéristiques compactes : torch.Size([1, 4096])
DataFrame avec caractéristiques compactes (4096) sauvegardé sous : E:\wealth_predict\data\processed_csv\DataCIV3_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv


In [2]:
pd.read_csv(r"E:\wealth_predict\data\processed_csv\DataCIV3_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv")

,nom de l'image,GPS__Latitude,GPS__Longitude,grappe,region,hhweight,hhsize,pcexp,country,year,...,feature_4086,feature_4087,feature_4088,feature_4089,feature_4090,feature_4091,feature_4092,feature_4093,feature_4094,feature_4095
0,lat_10.0003794_lon_-5.5802424_zoom_18.jpeg,10.000379,-5.580242,1047,TCHOLOGO,268.36096,4,274696.28,CIV,2018,...,0.050591,0.163085,0.059722,0.023969,0.003860,0.012237,0.051871,0.007524,0.028182,0.292025
1,lat_10.000821_lon_-5.5790777_zoom_18.jpeg,10.000821,-5.579078,1047,TCHOLOGO,268.36096,7,509182.78,CIV,2018,...,0.072871,0.193826,0.058918,0.024232,0.001967,0.016194,0.004346,0.008695,0.092535,0.367593
2,lat_10.0019714981318_lon_-5.93793609179556_zoo...,10.001971,-5.937936,168,PORO,337.44418,9,209861.88,CIV,2019,...,0.008498,0.410319,0.035861,0.077639,0.014966,0.004212,0.027701,0.017871,0.066511,0.003921
3,lat_10.0020737_lon_-5.5787912_zoom_18.jpeg,10.002074,-5.578791,1047,TCHOLOGO,268.36096,2,1117854.80,CIV,2018,...,0.094488,0.132202,0.108495,0.022925,0.005538,0.022400,0.024137,0.004544,0.044401,0.300378
4,lat_10.0052188_lon_-5.9011314_zoom_18.jpeg,10.005219,-5.901131,167,PORO,547.83280,1,695713.06,CIV,2018,...,0.065747,0.130906,0.138436,0.018007,0.004364,0.000874,0.056489,0.002114,0.074672,0.260308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12975,lat_9.9978309_lon_-5.5790889_zoom_18.jpeg,9.997831,-5.579089,1047,TCHOLOGO,268.36096,2,831854.94,CIV,2018,...,0.080837,0.083873,0.057509,0.045789,0.031724,0.030280,0.022594,0.005046,0.076345,0.256740
12976,lat_9.9984151404351_lon_-5.579286608845_zoom_1...,9.998415,-5.579287,1047,TCHOLOGO,268.36096,3,593999.44,CIV,2018,...,0.115947,0.113101,0.106721,0.035795,0.016075,0.019587,0.016419,0.012596,0.102024,0.451441
12977,lat_9.998664_lon_-5.5785752_zoom_18.jpeg,9.998664,-5.578575,1047,TCHOLOGO,268.36096,10,279080.53,CIV,2018,...,0.106856,0.149136,0.117501,0.065388,0.048576,0.028066,0.052733,0.019443,0.056493,0.325248
12978,lat_9.9990613_lon_-5.5788409_zoom_18.jpeg,9.999061,-5.578841,1047,TCHOLOGO,268.36096,1,1133500.60,CIV,2018,...,0.123448,0.154928,0.125896,0.051608,0.025298,0.026040,0.027721,0.008080,0.049624,0.426610


### Les caractéristiques sont condensées en un vecteur compact grâce à la réduction spatiale. Cela est utile pour des tâches de classification directe.
#### celà produit un vecteur avec 4 colonnes au lieu de 4096, les colonnes représentent les classes un peu comme s'il donne des scores d'appartenir à une classe

In [3]:
'''
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models

# Chemins des fichiers et répertoires
model_path = r"D:\wealth_predict_test\models\model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt"
test_image_dir = r"D:\wealth_predict_test\data\downloaded\Image_satellite_EHCVM_2018_Zoom_18_Image_2024"
csv_path = r"D:\wealth_predict_test\data\processed_csv\DataCIV3_images_with_new_columns.csv"
output_path = r"D:\wealth_predict_test\data\processed_csv\DataCIV3_images_with_features_extracted_fullyConv_sans_augm_couche_nongelee_batch_16.csv"

# Définir l'architecture du modèle
class VGGFullyConv(nn.Module):
    def __init__(self, num_classes=4):
        super(VGGFullyConv, self).__init__()
        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, num_classes, kernel_size=1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.mean(dim=(2, 3))  # Réduction spatiale

# Charger le DataFrame contenant les métadonnées
df = pd.read_csv(csv_path)

# Charger le modèle sauvegardé
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VGGFullyConv(num_classes=4).to(device)  # Recréer l'architecture
model.load_state_dict(torch.load(model_path))  # Charger les poids
model.eval()  # Mettre le modèle en mode évaluation

# Transformation des images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Fonction pour charger une image
def load_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        return transform(image)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

# Vérification dynamique du nombre de caractéristiques
dummy_image = torch.randn(1, 3, 224, 224).to(device)  # Image factice
dummy_features = model(dummy_image)  # Passer l'image dans le modèle
num_features = dummy_features.shape[1]  # Nombre de caractéristiques extraites

# Extraction des caractéristiques
def extract_image_features(model, image_tensor):
    image_tensor = image_tensor.unsqueeze(0).to(device)  # Ajouter une dimension pour le batch
    with torch.no_grad():
        features = model(image_tensor)  # Passer l'image dans le modèle
    return features.flatten().cpu().numpy()  # Retourner les caractéristiques comme tableau numpy

# Liste pour stocker les caractéristiques des images
features_list = []
for idx, row in df.iterrows():
    image_name = row['nom de l\'image']
    image_path = os.path.join(test_image_dir, image_name)

    if os.path.exists(image_path):  # Vérifier si l'image existe
        image_tensor = load_image(image_path)
        if image_tensor is not None:
            features = extract_image_features(model, image_tensor)
        else:
            features = np.zeros(num_features)  # Vecteur par défaut si erreur
    else:
        print(f"Image introuvable : {image_name}")
        features = np.zeros(num_features)  # Vecteur par défaut si l'image n'existe pas

    features_list.append(features)

# Conversion des caractéristiques en DataFrame
features_df = pd.DataFrame(features_list, columns=[f"feature_{i}" for i in range(num_features)])

# Combiner les caractéristiques avec le DataFrame original
df_with_features = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# Sauvegarder le DataFrame final
df_with_features.to_csv(output_path, index=False)
print(f"DataFrame avec caractéristiques sauvegardé sous : {output_path}")
'''

'\nimport os\nimport pandas as pd\nimport torch\nfrom torchvision import transforms\nfrom PIL import Image\nimport numpy as np\nfrom torch import nn\nfrom torchvision import models\n\n# Chemins des fichiers et répertoires\nmodel_path = r"D:\\wealth_predict_test\\models\\model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt"\ntest_image_dir = r"D:\\wealth_predict_test\\data\\downloaded\\Image_satellite_EHCVM_2018_Zoom_18_Image_2024"\ncsv_path = r"D:\\wealth_predict_test\\data\\processed_csv\\DataCIV3_images_with_new_columns.csv"\noutput_path = r"D:\\wealth_predict_test\\data\\processed_csv\\DataCIV3_images_with_features_extracted_fullyConv_sans_augm_couche_nongelee_batch_16.csv"\n\n# Définir l\'architecture du modèle\nclass VGGFullyConv(nn.Module):\n    def __init__(self, num_classes=4):\n        super(VGGFullyConv, self).__init__()\n        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features\n        self.classifier = nn.Sequential(\n            nn.C

In [4]:
#import pandas as pd
#pd.read_csv(r'D:\wealth_predict_test\data\processed_csv\DataCIV3_images_with_features_extracted_fullyConv_sans_augm_couche_nongelee_batch_16.csv')